In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("milestone3").getOrCreate()

from google.colab import files
uploaded = files.upload()

dataset_path = "/content/fintech_data_23_52_21240.parquet"

df = spark.read.parquet(dataset_path)
df.show(20)

Saving fintech_data_23_52_21240.parquet to fintech_data_23_52_21240.parquet
+--------------------+--------------------+----------+--------------+----------+----------------+-------------------+--------+----------+-----------+-----------+-------+-----------+-----------+-----+-------------+----------+--------+-----+-----------------+----------+----------+------------------+--------------------+
|         Customer Id|           Emp Title|Emp Length|Home Ownership|Annual Inc|Annual Inc Joint|Verification Status|Zip Code|Addr State|Avg Cur Bal|Tot Cur Bal|Loan Id|Loan Status|Loan Amount|State|Funded Amount|      Term|Int Rate|Grade|       Issue Date|Pymnt Plan|      Type|           Purpose|         Description|
+--------------------+--------------------+----------+--------------+----------+----------------+-------------------+--------+----------+-----------+-----------+-------+-----------+-----------+-----+-------------+----------+--------+-----+-----------------+----------+----------+-----

In [ ]:
print(df.rdd.getNumPartitions()) #check how many partitions

logical_cores = spark.sparkContext.defaultParallelism #this is the number of logical cores allocated to our colab VM since we are not running on local machine
df = df.repartition(logical_cores)

print(df.rdd.getNumPartitions()) #check new partitions

1
2


In [ ]:
from pyspark.sql.functions import col

df = df.select([col(c).alias(c.replace(" ", "_").lower()) for c in df.columns])
df.show(1)

+--------------------+--------------------+----------+--------------+----------+----------------+-------------------+--------+----------+-----------+-----------+-------+-----------+-----------+-----+-------------+----------+--------+-----+------------+----------+----------+------------------+------------------+
|         customer_id|           emp_title|emp_length|home_ownership|annual_inc|annual_inc_joint|verification_status|zip_code|addr_state|avg_cur_bal|tot_cur_bal|loan_id|loan_status|loan_amount|state|funded_amount|      term|int_rate|grade|  issue_date|pymnt_plan|      type|           purpose|       description|
+--------------------+--------------------+----------+--------------+----------+----------------+-------------------+--------+----------+-----------+-----------+-------+-----------+-----------+-----+-------------+----------+--------+-----+------------+----------+----------+------------------+------------------+
|YiJceGEyRCdkXHgxY...|Armed Security Su...|   9 years|       

In [ ]:
from pyspark.sql.functions import isnan, when

def calculate_missing_percentage(df):
  total_rows = df.count()
  missing_info = {}

  for column_name in df.columns:
      column_type = dict(df.dtypes)[column_name]

      if column_type in ['double', 'float', 'int', 'bigint', 'long']: #if it is a numerical column we check isNull and isnan
          missing_count = df.filter((col(column_name).isNull()) | isnan(col(column_name))).count()
      else:
          missing_count = df.filter(col(column_name).isNull()).count() #otherwise we check isNull only

      missing_percentage = (missing_count / total_rows) * 100
      missing_info[column_name] = missing_percentage

  return missing_info

missing_data = calculate_missing_percentage(df)
print(missing_data)

{'customer_id': 0.0, 'emp_title': 8.912319644839068, 'emp_length': 7.06992230854606, 'home_ownership': 0.0, 'annual_inc': 0.0, 'annual_inc_joint': 93.21124676285608, 'verification_status': 0.0, 'zip_code': 0.0, 'addr_state': 0.0, 'avg_cur_bal': 0.0, 'tot_cur_bal': 0.0, 'loan_id': 0.0, 'loan_status': 0.0, 'loan_amount': 0.0, 'state': 0.0, 'funded_amount': 0.0, 'term': 0.0, 'int_rate': 4.491305956344802, 'grade': 0.0, 'issue_date': 0.0, 'pymnt_plan': 0.0, 'type': 0.0, 'purpose': 0.0, 'description': 0.8509064002959674}


In [ ]:
#we have missing data in the following numerical columns: annual_inc_joint and int_rate
# and missing data in the following categorical: emp_title, emp_length and description

#starting with numerical
df = df.fillna(value=0, subset=["annual_inc_joint"])
df = df.fillna(value=0, subset=["int_rate"])

#then categorical
mode_title = df.groupBy('emp_title').count().orderBy(col('count').desc()).limit(1)
mode_length = df.groupBy('emp_length').count().orderBy(col('count').desc()).limit(1)
mode_description = df.groupBy('description').count().orderBy(col('count').desc()).limit(1)

mode_title = mode_title.select('emp_title').collect()[0][0]
mode_length = mode_length.select('emp_length').collect()[0][0]
mode_description = mode_description.select('description').collect()[0][0]

print(mode_title, mode_length, mode_description) #mode_title is None

#we will get the second most frequent value and use it as a mode instead
mode_title = df.groupBy('emp_title').count().orderBy(col('count').desc()).limit(2)
mode_title = mode_title.select('emp_title').collect()[1][0]

print(mode_title) #mode_title is not None

df = df.fillna(value=mode_title, subset=['emp_title'])
df = df.fillna(value=mode_length, subset=['emp_length'])
df = df.fillna(value=mode_description, subset=['description'])

print(df.filter(df.annual_inc_joint.isNull()).count(), df.filter(df.int_rate.isNull()).count(), df.filter(df.emp_title.isNull()).count(),
      df.filter(df.emp_length.isNull()).count(), df.filter(df.description.isNull()).count()) #now we have no missing data

None 10+ years Debt consolidation
Teacher
0 0 0 0 0


In [29]:
from pyspark.sql.functions import regexp_replace

#emp_length encoding
df_encoded = df.withColumn("emp_length", regexp_replace("emp_length", "10\\+ years", "10")) #assume more than 10 years is 10
df_encoded = df_encoded.withColumn("emp_length", regexp_replace("emp_length", "< 1 year", "0")) #assume less than 1 year is 0
df_encoded = df_encoded.withColumn("emp_length", regexp_replace("emp_length", " years?", ""))
df_encoded = df_encoded.withColumn("emp_length", regexp_replace("emp_length", " year", ""))
df.select('emp_length').show(5)
df_encoded.select('emp_length').show(5)

#create emp_length_mapping_df to be used in lookup table
emp_length_mapping = {
    "10+ years": 10,
    "< 1 year": 0,
    "1 year": 1,
    "2 years": 2,
    "3 years": 3,
    "4 years": 4,
    "5 years": 5,
    "6 years": 6,
    "7 years": 7,
    "8 years": 8,
    "9 years": 9
}

emp_length_mapping_table = [(f"emp_length {key}", value) for key, value in emp_length_mapping.items()]
emp_length_mapping_df = spark.createDataFrame(emp_length_mapping_table, ["feature", "encoded_value"])

emp_length_mapping_df.show()

+----------+
|emp_length|
+----------+
|   5 years|
|   5 years|
|   5 years|
|   5 years|
|   5 years|
+----------+
only showing top 5 rows

+----------+
|emp_length|
+----------+
|         5|
|         5|
|         5|
|         5|
|         5|
+----------+
only showing top 5 rows

+--------------------+-------------+
|             feature|encoded_value|
+--------------------+-------------+
|emp_length 10+ years|           10|
| emp_length < 1 year|            0|
|   emp_length 1 year|            1|
|  emp_length 2 years|            2|
|  emp_length 3 years|            3|
|  emp_length 4 years|            4|
|  emp_length 5 years|            5|
|  emp_length 6 years|            6|
|  emp_length 7 years|            7|
|  emp_length 8 years|            8|
|  emp_length 9 years|            9|
+--------------------+-------------+



In [30]:
from pyspark.ml.feature import StringIndexer

def label_encoding(df, column):
  indexer = StringIndexer(inputCol=column, outputCol=column+"_encoded")
  indexer_model = indexer.fit(df)
  df_encoded = indexer_model.transform(df)
  df_encoded = df_encoded.withColumn(column, df_encoded[f"{column}_encoded"].cast("int")) #encoding done in place since it is label encoding
  df_encoded = df_encoded.drop(f"{column}_encoded") #drop the extra column
  #used for lookup table
  mapping = indexer_model.labels
  mapping_table = [(f"{column} {label}", index) for index, label in enumerate(mapping)]
  mapping_df = spark.createDataFrame(mapping_table, ["feature", "encoded_value"])
  return df_encoded, mapping_df

def one_hot_encoding(df, column):
  unique_values = df.select(column).distinct().rdd.flatMap(lambda x: x).collect()
  for value in unique_values:
    new_column = f"{column}_{value}"
    df = df.withColumn(new_column, when(col(column) == value, 1).otherwise(0))
  #used for lookup table
  mapping_table = [(f"{column} {value}", "one hot encoded") for value in unique_values]
  mapping_df = spark.createDataFrame(mapping_table, ["feature", "encoded_value"])

  df = df.drop(column) #drop the original column
  return df, mapping_df

def discretize_grade(df):
  df = df.withColumn("grade",
    when((col("grade") >= 1) & (col("grade") <= 5), "A")
    .when((col("grade") >= 6) & (col("grade") <= 10), "B")
    .when((col("grade") >= 11) & (col("grade") <= 15), "C")
    .when((col("grade") >= 16) & (col("grade") <= 20), "D")
    .when((col("grade") >= 21) & (col("grade") <= 25), "E")
    .when((col("grade") >= 26) & (col("grade") <= 30), "F")
    .when((col("grade") >= 31) & (col("grade") <= 35), "G")
    )
  #used for lookup table
  grade_mapping = {
      "A": "1 - 5",
      "B": "6 - 10",
      "C": "11 - 15",
      "D": "16 - 20",
      "E": "21 - 25",
      "F": "26 - 30",
      "G": "31 - 35"
  }
  mapping_table = [(f"grade {key}", value) for key, value in grade_mapping.items()]
  mapping_df = spark.createDataFrame(mapping_table, ["feature", "encoded_value"])
  return df, mapping_df

#label encoding columns: state and purpose
df_encoded, state_mapping_df = label_encoding(df_encoded, "state")
df_encoded, purpose_mapping_df = label_encoding(df_encoded, "purpose")

#one hot encoding columns: home_ownership, verification_status and type
df_encoded, home_ownership_mapping_df = one_hot_encoding(df_encoded, "home_ownership")
df_encoded, verification_status_mapping_df = one_hot_encoding(df_encoded, "verification_status")
df_encoded, type_mapping_df = one_hot_encoding(df_encoded, "type")

#for the grade column
df_encoded, grade_mapping_df = discretize_grade(df_encoded)

df_encoded.show(10)
state_mapping_df.show(10)
purpose_mapping_df.show()

+--------------------+--------------------+----------+----------+----------------+--------+----------+-----------+-----------+-------+-----------+-----------+-----+-------------+----------+--------+-----+---------------+----------+-------+--------------------+------------------+-------------------+-----------------------+------------------+----------------------------+-----------------------------------+--------------------------------+--------------+---------------+---------------+----------+
|         customer_id|           emp_title|emp_length|annual_inc|annual_inc_joint|zip_code|addr_state|avg_cur_bal|tot_cur_bal|loan_id|loan_status|loan_amount|state|funded_amount|      term|int_rate|grade|     issue_date|pymnt_plan|purpose|         description|home_ownership_OWN|home_ownership_RENT|home_ownership_MORTGAGE|home_ownership_ANY|verification_status_Verified|verification_status_Source Verified|verification_status_Not Verified|type_Joint App|type_INDIVIDUAL|type_DIRECT_PAY|type_JOINT|
+-

In [31]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def add_features(df):
  #we create a temp df to perform operations on, in order not to manipulate row order in the original df
  df_temp = df.withColumn("issue_date", F.to_date("issue_date", "dd MMMM yyyy"))

  #window specification partitioned by grade and ordered by issue_date
  window_grade = Window.partitionBy("grade").orderBy("issue_date")
  window_state_grade = Window.partitionBy("state", "grade").orderBy("issue_date")

  df_temp = df_temp.withColumn("prev_loan_date_same_grade", F.lag("issue_date", 1).over(window_grade))
  df_temp = df_temp.withColumn("prev_loan_amt_same_grade", F.lag("loan_amount", 1).over(window_grade))
  df_temp = df_temp.withColumn("prev_loan_date_same_state_grade", F.lag("issue_date", 1).over(window_state_grade))
  df_temp = df_temp.withColumn("prev_loan_amt_same_state_grade", F.lag("loan_amount", 1).over(window_state_grade))

  #we select these features to join them to the original df, matching on loan_id
  features = df_temp.select("loan_id",
                            "prev_loan_date_same_grade",
                            "prev_loan_amt_same_grade",
                            "prev_loan_date_same_state_grade",
                            "prev_loan_amt_same_state_grade"
                            )

  df = df.join(features, on="loan_id", how="left")

  return df

df_encoded = add_features(df_encoded)
df_encoded.show(10)

+-------+--------------------+--------------------+----------+----------+----------------+--------+----------+-----------+-----------+-----------+-----------+-----+-------------+----------+--------+-----+---------------+----------+-------+--------------------+------------------+-------------------+-----------------------+------------------+----------------------------+-----------------------------------+--------------------------------+--------------+---------------+---------------+----------+-------------------------+------------------------+-------------------------------+------------------------------+
|loan_id|         customer_id|           emp_title|emp_length|annual_inc|annual_inc_joint|zip_code|addr_state|avg_cur_bal|tot_cur_bal|loan_status|loan_amount|state|funded_amount|      term|int_rate|grade|     issue_date|pymnt_plan|purpose|         description|home_ownership_OWN|home_ownership_RENT|home_ownership_MORTGAGE|home_ownership_ANY|verification_status_Verified|verification_sta

In [32]:
df_encoded.createOrReplaceTempView("fintech") #to execute sql queries

#query 1
query1 = spark.sql("""
    SELECT
      emp_length,
      CASE
        WHEN annual_inc < 50000 THEN '<50k'
        WHEN annual_inc BETWEEN 50000 AND 100000 THEN '50k-100k'
        ELSE '>100k'
      END AS income_range,
      AVG(loan_amount) AS avg_loan_amount,
      AVG(int_rate) AS avg_int_rate
    FROM fintech
    WHERE loan_status = 'Current'
    GROUP BY emp_length, income_range
    ORDER BY emp_length, income_range;

""")

query1.show()

+----------+------------+------------------+-------------------+
|emp_length|income_range|   avg_loan_amount|       avg_int_rate|
+----------+------------+------------------+-------------------+
|         0|    50k-100k|15829.424778761062|0.11772187104930475|
|         0|        <50k|10875.623885918003|0.12575436720142597|
|         0|       >100k|23089.036312849163|0.11115782122905038|
|         1|    50k-100k|14844.649446494464|0.12145239852398525|
|         1|        <50k|10016.739766081871| 0.1281649122807018|
|         1|       >100k|23276.973684210527|0.11150701754385968|
|        10|    50k-100k| 16507.19572368421|0.12269967105263152|
|        10|        <50k| 10874.20353982301|0.13049085545722705|
|        10|       >100k|22681.738849385907|0.11220678733031669|
|         2|    50k-100k|15021.295103092783|0.12159458762886598|
|         2|        <50k|  9697.86432160804| 0.1330979899497488|
|         2|       >100k|21982.864238410595|0.11060860927152324|
|         3|    50k-100k|

In [33]:
from pyspark.sql.functions import avg

#query 1 but with Spark
#defining bins
df_encoded = df_encoded.withColumn(
    "income_range",
    when(col("annual_inc") < 50000, "<50k")
    .when((col("annual_inc") >= 50000) & (col("annual_inc") <= 100000), "50k-100k")
    .otherwise(">100k")
)

#selecting Current loan status
default_loans = df_encoded.filter(col("loan_status") == "Current")

result1 = default_loans.groupBy("emp_length", "income_range").agg(
    avg("loan_amount").alias("avg_loan_amount"),
    avg("int_rate").alias("avg_int_rate")
).orderBy("emp_length", "income_range")

result1.show()

#drop the temporarily added column for the query "income_range"
df_encoded = df_encoded.drop("income_range")

+----------+------------+------------------+-------------------+
|emp_length|income_range|   avg_loan_amount|       avg_int_rate|
+----------+------------+------------------+-------------------+
|         0|    50k-100k|15829.424778761062|0.11772187104930475|
|         0|        <50k|10875.623885918003|0.12575436720142597|
|         0|       >100k|23089.036312849163|0.11115782122905038|
|         1|    50k-100k|14844.649446494464|0.12145239852398525|
|         1|        <50k|10016.739766081871| 0.1281649122807018|
|         1|       >100k|23276.973684210527|0.11150701754385968|
|        10|    50k-100k| 16507.19572368421|0.12269967105263152|
|        10|        <50k| 10874.20353982301|0.13049085545722705|
|        10|       >100k|22681.738849385907|0.11220678733031669|
|         2|    50k-100k|15021.295103092783|0.12159458762886598|
|         2|        <50k|  9697.86432160804| 0.1330979899497488|
|         2|       >100k|21982.864238410595|0.11060860927152324|
|         3|    50k-100k|

In [34]:
#query 2
query2 = spark.sql("""
    SELECT
      grade,
      AVG(loan_amount - funded_amount) AS avg_diff
    FROM fintech
    GROUP BY grade
    ORDER BY avg_diff DESC;
""")

query2.show()

+-----+--------+
|grade|avg_diff|
+-----+--------+
|    F|     0.0|
|    E|     0.0|
|    B|     0.0|
|    D|     0.0|
|    C|     0.0|
|    A|     0.0|
|    G|     0.0|
+-----+--------+



In [35]:
#calculate difference and group by grade
result2 = df_encoded.withColumn(
    "amount_diff", col("loan_amount") - col("funded_amount")
).groupBy("grade").agg(
    avg("amount_diff").alias("avg_diff")
).orderBy(col("avg_diff").desc())

result2.show()

+-----+--------+
|grade|avg_diff|
+-----+--------+
|    F|     0.0|
|    E|     0.0|
|    B|     0.0|
|    D|     0.0|
|    C|     0.0|
|    A|     0.0|
|    G|     0.0|
+-----+--------+



In [36]:
df.createOrReplaceTempView("fintech2") #this query needs verification_status column which was encoded earlier so we use df instead of df_encoded

#query 3
query3 = spark.sql("""
    SELECT
      addr_state,
      verification_status,
      SUM(loan_amount) AS total_loan_amount
    FROM fintech2
    WHERE verification_status IN ('Verified', 'Not Verified')
    GROUP BY addr_state, verification_status
    ORDER BY addr_state, verification_status;
""")

query3.show()

+----------+-------------------+-----------------+
|addr_state|verification_status|total_loan_amount|
+----------+-------------------+-----------------+
|        AK|       Not Verified|         280225.0|
|        AK|           Verified|         354425.0|
|        AL|       Not Verified|        1326300.0|
|        AL|           Verified|        1598025.0|
|        AR|       Not Verified|        1094350.0|
|        AR|           Verified|         861925.0|
|        AZ|       Not Verified|        2919700.0|
|        AZ|           Verified|        2490325.0|
|        CA|       Not Verified|       1.773405E7|
|        CA|           Verified|       1.657705E7|
|        CO|       Not Verified|        2816400.0|
|        CO|           Verified|        2780625.0|
|        CT|       Not Verified|        2063125.0|
|        CT|           Verified|        1721475.0|
|        DC|       Not Verified|         312875.0|
|        DC|           Verified|         179925.0|
|        DE|       Not Verified

In [37]:
from pyspark.sql.functions import sum

#filter for Verified and Not Verified only, exclude Source Verified
df_filter = df.filter(col("verification_status").isin("Verified", "Not Verified"))

#group by addr_state and verification_status then calculate total loan amount
result3 = df_filter.groupBy("addr_state", "verification_status").agg(
    sum("loan_amount").alias("total_loan_amount")
).orderBy("addr_state", "verification_status")

result3.show()

+----------+-------------------+-----------------+
|addr_state|verification_status|total_loan_amount|
+----------+-------------------+-----------------+
|        AK|       Not Verified|         280225.0|
|        AK|           Verified|         354425.0|
|        AL|       Not Verified|        1326300.0|
|        AL|           Verified|        1598025.0|
|        AR|       Not Verified|        1094350.0|
|        AR|           Verified|         861925.0|
|        AZ|       Not Verified|        2919700.0|
|        AZ|           Verified|        2490325.0|
|        CA|       Not Verified|       1.773405E7|
|        CA|           Verified|       1.657705E7|
|        CO|       Not Verified|        2816400.0|
|        CO|           Verified|        2780625.0|
|        CT|       Not Verified|        2063125.0|
|        CT|           Verified|        1721475.0|
|        DC|       Not Verified|         312875.0|
|        DC|           Verified|         179925.0|
|        DE|       Not Verified

In [38]:
from pyspark.sql.functions import to_date

df_temp = df_encoded.withColumn("issue_date", to_date("issue_date", "d MMMM yyyy")) \
                                   .withColumn("prev_loan_date_same_grade", to_date("prev_loan_date_same_grade", "d MMMM yyyy"))\
                                   .withColumn("prev_loan_date_same_state_grade", to_date("prev_loan_date_same_state_grade", "d MMMM yyyy"))

df_temp.createOrReplaceTempView("fintech3")

#query 4
query4 = spark.sql("""
    SELECT
      grade,
      AVG(DATEDIFF(issue_date, prev_loan_date_same_grade)) AS avg_gap_days
    FROM fintech3
    WHERE prev_loan_date_same_grade IS NOT NULL
    GROUP BY grade
    ORDER BY avg_gap_days
""")

query4.show()

+-----+-------------------+
|grade|       avg_gap_days|
+-----+-------------------+
|    B|0.33608711978971084|
|    C|0.35301078096239813|
|    A| 0.4655800242760534|
|    D| 0.6902313624678663|
|    E|  1.891679748822606|
|    F|  6.254847645429363|
|    G| 15.470588235294118|
+-----+-------------------+



In [39]:
from pyspark.sql.functions import datediff

result4 = df_temp.withColumn(
    "gap_days", datediff(col("issue_date"), col("prev_loan_date_same_grade"))
).groupBy("grade").agg(
    avg("gap_days").alias("avg_gap_days")
).orderBy("avg_gap_days")

result4.show()

+-----+-------------------+
|grade|       avg_gap_days|
+-----+-------------------+
|    B|0.33608711978971084|
|    C|0.35301078096239813|
|    A| 0.4655800242760534|
|    D| 0.6902313624678663|
|    E|  1.891679748822606|
|    F|  6.254847645429363|
|    G| 15.470588235294118|
+-----+-------------------+



In [40]:
#query 5
query5= spark.sql("""
    SELECT
      state,
      grade,
      AVG(loan_amount - prev_loan_amt_same_state_grade) AS avg_amount_diff
    FROM fintech
    GROUP BY state, grade
    ORDER BY state, grade;
""")

query5.show()

+-----+-----+-------------------+
|state|grade|    avg_amount_diff|
+-----+-----+-------------------+
|    0|    A|  3.594351732991014|
|    0|    B|  30.49645390070922|
|    0|    C| 24.019607843137255|
|    0|    D|-14.299065420560748|
|    0|    E|-140.74675324675326|
|    0|    F|-106.38297872340425|
|    0|    G| 1133.3333333333333|
|    1|    A| 20.242914979757085|
|    1|    B|   9.79020979020979|
|    1|    C|-1.0738831615120275|
|    1|    D| -34.59119496855346|
|    1|    E|  52.41935483870968|
|    1|    F| 312.14285714285717|
|    1|    G|  1888.888888888889|
|    2|    A| 4.9281314168377826|
|    2|    B|-10.014727540500736|
|    2|    C| -9.950248756218905|
|    2|    D|-14.482758620689655|
|    2|    E| -70.79646017699115|
|    2|    F|-272.72727272727275|
+-----+-----+-------------------+
only showing top 20 rows



In [41]:
#calculate loan amount differenc and group by state and grade
result5 = df_encoded.withColumn(
    "amount_diff", col("loan_amount") - col("prev_loan_amt_same_state_grade")
).groupBy("state", "grade").agg(
    avg("amount_diff").alias("avg_amount_diff")
).orderBy("state", "grade")

result5.show()

+-----+-----+-------------------+
|state|grade|    avg_amount_diff|
+-----+-----+-------------------+
|    0|    A|  3.594351732991014|
|    0|    B|  30.49645390070922|
|    0|    C| 24.019607843137255|
|    0|    D|-14.299065420560748|
|    0|    E|-140.74675324675326|
|    0|    F|-106.38297872340425|
|    0|    G| 1133.3333333333333|
|    1|    A| 20.242914979757085|
|    1|    B|   9.79020979020979|
|    1|    C|-1.0738831615120275|
|    1|    D| -34.59119496855346|
|    1|    E|  52.41935483870968|
|    1|    F| 312.14285714285717|
|    1|    G|  1888.888888888889|
|    2|    A| 4.9281314168377826|
|    2|    B|-10.014727540500736|
|    2|    C| -9.950248756218905|
|    2|    D|-14.482758620689655|
|    2|    E| -70.79646017699115|
|    2|    F|-272.72727272727275|
+-----+-----+-------------------+
only showing top 20 rows



In [42]:
#use mapping dfs to form lookup table
def form_lookup(mapping_dfs):
  lookup_df = mapping_dfs[0]
  for mapping_df in mapping_dfs[1:]:
    lookup_df = lookup_df.unionByName(mapping_df, allowMissingColumns=True)

  return lookup_df

#all mapping dfs defined previously
mapping_dfs = [emp_length_mapping_df, state_mapping_df, purpose_mapping_df, home_ownership_mapping_df, verification_status_mapping_df, type_mapping_df, grade_mapping_df]

lookup_df = form_lookup(mapping_dfs)
lookup_df.show()

+--------------------+-------------+
|             feature|encoded_value|
+--------------------+-------------+
|emp_length 10+ years|           10|
| emp_length < 1 year|            0|
|   emp_length 1 year|            1|
|  emp_length 2 years|            2|
|  emp_length 3 years|            3|
|  emp_length 4 years|            4|
|  emp_length 5 years|            5|
|  emp_length 6 years|            6|
|  emp_length 7 years|            7|
|  emp_length 8 years|            8|
|  emp_length 9 years|            9|
|            state CA|            0|
|            state TX|            1|
|            state NY|            2|
|            state FL|            3|
|            state IL|            4|
|            state NJ|            5|
|            state PA|            6|
|            state GA|            7|
|            state OH|            8|
+--------------------+-------------+
only showing top 20 rows



In [43]:
#now saving both dfs (cleaned df and lookup table) to parquet files
df_encoded.write.mode("overwrite").parquet("/content/fintech_spark_52_21240_clean.parquet")
lookup_df.write.mode("overwrite").parquet("/content/lookup_spark_52_21240.parquet")

#mounting drive to colab then copying both parquet files in google drive
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/fintech_spark_52_21240_clean.parquet /content/drive/MyDrive/
!cp -r /content/lookup_spark_52_21240.parquet /content/drive/MyDrive/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
